# Week 10 AI-Assisted Structured Data Conversion Lab

In this notebook, you will compare the supplied CSV and JSON files, examine the original values, identify inconsistencies, and standardize selected fields. Keep the notebook, CSV file, and JSON file in the same folder. Use only the supplied synthetic data.

**Standardizing a value** means converting different representations of the same category or date into one approved representation. For example, `c01` and `C01` refer to the same clinic code, but the cleaned data should consistently use `C01`. Standardization does not mean inventing missing information or changing what a value means.

Complete each TODO in order. Run a cell before moving to the next section so you can use its output to guide your work.


In [1]:
from pathlib import Path
import csv
import json
from datetime import datetime
from collections import Counter

CSV_PATH = Path("Week10_Structured_Referrals.csv")
JSON_PATH = Path("Week10_Structured_Referrals.json")


## 1 Load and compare the files

Load each file as a **list of dictionaries**:

- `csv_records` should contain the records read from the CSV file with `csv.DictReader`.
- `json_records` should contain the records read from the JSON file with `json.load`.

The validation cell checks that both files contain 12 records and the same referral identifiers.


In [2]:
# TODO 1: Open CSV_PATH with newline="" and encoding="utf-8".
# Use csv.DictReader and convert the result to a list named csv_records.

# TODO 2: Open JSON_PATH with encoding="utf-8".
# Use json.load and store the result in json_records.

with open(CSV_PATH, mode="r", newline="", encoding="utf-8") as file:
    csv_records = list(csv.DictReader(file))
with open(JSON_PATH, mode="r", encoding="utf-8") as file:
    json_records = json.load(file)

print("CSV records:", len(csv_records))
print("JSON records:", len(json_records))


CSV records: 12
JSON records: 12


In [3]:
# Validation test: do not change this cell.
assert len(csv_records) == 12, "The CSV file should contain 12 records."
assert len(json_records) == 12, "The JSON file should contain 12 records."
assert {r["referral_id"] for r in csv_records} == {r["referral_id"] for r in json_records},     "The CSV and JSON referral identifiers should match."
print("File comparison tests passed.")


File comparison tests passed.


## 2 Preview the original data

Before cleaning data, examine what the files actually contain. Run the next cell to display:

- The field names in each format
- The first three CSV records
- The first three JSON records

Look for differences in capitalization, abbreviations, names used in place of codes, and date formats. Also notice how a missing completion date appears in each format.


In [4]:
print("CSV field names:", list(csv_records[0].keys()))
print("JSON field names:", list(json_records[0].keys()))

print("\nFirst three CSV records")
for number, record in enumerate(csv_records[:3], start=1):
    print(number, record)

print("\nFirst three JSON records")
for number, record in enumerate(json_records[:3], start=1):
    print(number, record)


CSV field names: ['referral_id', 'patient_id', 'clinic_code', 'service_type', 'priority', 'referral_date', 'due_date', 'completion_date']
JSON field names: ['referral_id', 'patient_id', 'clinic_code', 'service_type', 'priority', 'referral_date', 'due_date', 'completion_date']

First three CSV records
1 {'referral_id': 'R001', 'patient_id': 'P001', 'clinic_code': 'c01', 'service_type': 'Medication review', 'priority': 'routine', 'referral_date': '08/03/2026', 'due_date': '2026-08-17', 'completion_date': '2026-08-13'}
2 {'referral_id': 'R002', 'patient_id': 'P002', 'clinic_code': 'riverbend', 'service_type': 'social work', 'priority': 'Routine', 'referral_date': '2026-08-05', 'due_date': '2026-08-19', 'completion_date': '2026-08-18'}
3 {'referral_id': 'R003', 'patient_id': 'P003', 'clinic_code': 'c03', 'service_type': 'Nutrition', 'priority': 'High', 'referral_date': '2026-08-07', 'due_date': '2026-08-14', 'completion_date': '2026-08-14'}

First three JSON records
1 {'referral_id': 'R001

## 3 Identify values that are not standardized

The next cell lists the distinct values found in the fields you will clean. Distinct-value inspection is useful because inconsistent categories that look similar in a long file become easy to compare.

Run the cell, then complete the observation table below it. Do not begin writing the standardization functions until you have reviewed the output.


In [5]:
fields_to_inspect = [
    "clinic_code",
    "service_type",
    "priority",
    "referral_date",
    "due_date",
    "completion_date",
]

for field in fields_to_inspect:
    distinct_values = sorted({record[field] for record in csv_records})
    print(f"\n{field} ({len(distinct_values)} distinct values)")
    for value in distinct_values:
        print("  ", repr(value))



clinic_code (3 distinct values)
   'c01'
   'c03'
   'riverbend'

service_type (4 distinct values)
   'Care coordination'
   'Medication review'
   'Nutrition'
   'social work'

priority (4 distinct values)
   'High'
   'Routine'
   'high'
   'routine'

referral_date (12 distinct values)
   '08/03/2026'
   '08/09/2026'
   '08/15/2026'
   '08/21/2026'
   '2026-08-05'
   '2026-08-07'
   '2026-08-11'
   '2026-08-13'
   '2026-08-17'
   '2026-08-19'
   '2026-08-23'
   '2026-08-25'

due_date (12 distinct values)
   '2026-08-14'
   '2026-08-17'
   '2026-08-19'
   '2026-08-20'
   '2026-08-23'
   '2026-08-25'
   '2026-08-26'
   '2026-08-29'
   '2026-08-31'
   '2026-09-01'
   '2026-09-04'
   '2026-09-06'

completion_date (11 distinct values)
   ''
   '2026-08-13'
   '2026-08-14'
   '2026-08-18'
   '2026-08-21'
   '2026-08-23'
   '2026-08-26'
   '2026-08-31'
   '2026-09-01'
   '2026-09-03'
   '2026-09-05'


### Your observations

Replace the bracketed guidance after running the distinct-values cell.

| Field | Values or patterns you observed | Why the values are not yet standard |
|---|---|---|
| `clinic_code` | [List the distinct raw values] | [For example, lowercase codes or a clinic name instead of a code] |
| `service_type` | [List the distinct raw values] | [Identify inconsistent capitalization, if present] |
| `priority` | [List the distinct raw values] | [Identify labels that differ only by capitalization] |
| `referral_date` | [Describe both date patterns] | [Explain why mixed date formats can cause problems] |
| `completion_date` | [Describe the blank value] | [Explain what the blank means in this assignment] |

In 3–5 sentences, also explain why a list of dictionaries is useful for these records and identify one strength of CSV and one strength of JSON.

**Your response:** Using a list of dictionaries is a practical way to organize data records. It allows you to process information sequentially while keeping fields clearly labeled with descriptive names, rather than relying on numerical indexes.

For storing this data, the CSV format stands out for its simple, lightweight structure. It is highly efficient for standard tabular data and can be easily opened in everyday spreadsheet software.

However, for more complex datasets, JSON provides distinct advantages. It naturally preserves specific data types—such as numbers and true/false values—and handles nested, multi-layered information smoothly across different programming environments.


## 4 Standardization specifications

Use the following specifications exactly. The **raw values** are values you should have found when you inspected the original files. The **approved values** are the only values that should remain in the cleaned records.

### Clinic codes

| Raw value | Approved value | Clinic represented |
|---|---|---|
| `c01` | `C01` | Northside Clinic |
| `riverbend` | `C02` | Riverbend Clinic |
| `c03` | `C03` | Mountain View Clinic |

After standardization, `clinic_code` must contain only `C01`, `C02`, or `C03`.

### Service labels

Standardize capitalization so the cleaned `service_type` field contains only these exact labels:

- `Medication review`
- `Social work`
- `Nutrition`
- `Care coordination`

For example, `social work` becomes `Social work`. A label already written in an approved form should remain unchanged.

### Priority labels

The cleaned `priority` field must contain only:

- `High`
- `Routine`

For example, `high` becomes `High` and `routine` becomes `Routine`.

### Date fields

All nonmissing dates must use the ISO format `YYYY-MM-DD`.

- `referral_date` contains both `MM/DD/YYYY` and `YYYY-MM-DD` in the original files. Parse either format and return `YYYY-MM-DD`.
- `due_date` is already written as `YYYY-MM-DD`, but pass it through the same date function so the format is validated.
- `completion_date` should also use `YYYY-MM-DD` when it has a value.
- A blank `completion_date` means the referral has not been completed. Preserve that missing information as `None` in Python. Do not replace it with a date, zero, or the word `Unknown`.

Use `.strip()` to remove accidental spaces and case-insensitive matching when you look up category values. Your functions should not silently invent a result for an unexpected nonblank category.


## 5 Create standardization functions

Complete the four functions. The mapping dictionaries below define the approved results. Your functions should clean the incoming text, look it up in the appropriate mapping, and return the approved value.


In [6]:
CLINIC_MAP = {
    "c01": "C01",
    "riverbend": "C02",
    "c03": "C03",
}

SERVICE_MAP = {
    "medication review": "Medication review",
    "social work": "Social work",
    "nutrition": "Nutrition",
    "care coordination": "Care coordination",
}

PRIORITY_MAP = {
    "high": "High",
    "routine": "Routine",
}


def standardize_clinic(value):
    'Return C01, C02, or C03 for a known raw clinic value.'
    # TODO: remove surrounding spaces, convert to lowercase, and use CLINIC_MAP.
    cleaned_val = value.strip().lower()
    return CLINIC_MAP.get(cleaned_val)


def standardize_service(value):
    'Return one of the four approved service labels.'
    # TODO: remove surrounding spaces, convert to lowercase, and use SERVICE_MAP.
    cleaned_val = value.strip().lower()
    return SERVICE_MAP.get(cleaned_val)


def standardize_priority(value):
    'Return High or Routine for a known raw priority value.'
    # TODO: remove surrounding spaces, convert to lowercase, and use PRIORITY_MAP.
    cleaned_val = value.strip().lower()
    return PRIORITY_MAP.get(cleaned_val)


def standardize_date(value):
    'Accept YYYY-MM-DD or MM/DD/YYYY and return YYYY-MM-DD.'
    # TODO: remove surrounding spaces.
    # TODO: try datetime.strptime with "%Y-%m-%d" and "%m/%d/%Y".
    # TODO: return the parsed date with .date().isoformat().
    val = value.strip()
    try:
        return datetime.strptime(val, "%Y-%m-%d").date().isoformat()
    except ValueError:
        return datetime.strptime(val, "%m/%d/%Y").date().isoformat()


In [7]:
# Required function tests: do not change the expected values.
assert standardize_clinic("c01") == "C01"
assert standardize_clinic("riverbend") == "C02"
assert standardize_clinic("c03") == "C03"

assert standardize_service("social work") == "Social work"
assert standardize_service("Nutrition") == "Nutrition"
assert standardize_service("Care coordination") == "Care coordination"

assert standardize_priority("high") == "High"
assert standardize_priority("routine") == "Routine"
assert standardize_priority("High") == "High"

assert standardize_date("08/03/2026") == "2026-08-03"
assert standardize_date("2026-08-05") == "2026-08-05"
print("Function tests passed.")


Function tests passed.


## 6 Clean every record

Create a new list named `cleaned_records`. Do not edit `csv_records` in place.

For each original record:

1. Make a new dictionary with `cleaned = dict(record)`.
2. Standardize `clinic_code`, `service_type`, `priority`, `referral_date`, and `due_date`.
3. If `completion_date` is blank after stripping spaces, store `None`. Otherwise, standardize the date.
4. Append the new dictionary to `cleaned_records`.


In [8]:
cleaned_records = []

for record in csv_records:
    # TODO: make a copy of record.
    cleaned = dict(record)
    # TODO: replace the selected fields in the copied dictionary with standardized values.
    cleaned["clinic_code"] = standardize_clinic(cleaned["clinic_code"])
    cleaned["service_type"] = standardize_service(cleaned["service_type"])
    cleaned["priority"] = standardize_priority(cleaned["priority"])
    cleaned["referral_date"] = standardize_date(cleaned["referral_date"])
    cleaned["due_date"] = standardize_date(cleaned["due_date"])
    # TODO: handle a blank completion_date as None.
    if cleaned["completion_date"].strip() == "":
        cleaned["completion_date"] = None
    # TODO: append the cleaned dictionary to cleaned_records.
    cleaned_records.append(cleaned)
    pass

print("Cleaned records:", len(cleaned_records))
print("First two cleaned records:")
for record in cleaned_records[:2]:
    print(record)


Cleaned records: 12
First two cleaned records:
{'referral_id': 'R001', 'patient_id': 'P001', 'clinic_code': 'C01', 'service_type': 'Medication review', 'priority': 'Routine', 'referral_date': '2026-08-03', 'due_date': '2026-08-17', 'completion_date': '2026-08-13'}
{'referral_id': 'R002', 'patient_id': 'P002', 'clinic_code': 'C02', 'service_type': 'Social work', 'priority': 'Routine', 'referral_date': '2026-08-05', 'due_date': '2026-08-19', 'completion_date': '2026-08-18'}


In [9]:
# Required cleaned-data validation: do not change this cell.
assert len(cleaned_records) == 12

clinic_counts = Counter(r["clinic_code"] for r in cleaned_records)
assert clinic_counts == Counter({"C01": 4, "C02": 4, "C03": 4})

assert {r["service_type"] for r in cleaned_records} == {
    "Medication review", "Social work", "Nutrition", "Care coordination"
}
assert {r["priority"] for r in cleaned_records} == {"High", "Routine"}
assert all(len(r["referral_date"]) == 10 and r["referral_date"][4] == "-" for r in cleaned_records)
assert all(len(r["due_date"]) == 10 and r["due_date"][4] == "-" for r in cleaned_records)
assert any(r["completion_date"] is None for r in cleaned_records)

# Confirm that the original raw list was not edited in place.
assert csv_records[0]["clinic_code"] == "c01"

print("Cleaned-data tests passed.")
print("Clinic counts:", clinic_counts)


Cleaned-data tests passed.
Clinic counts: Counter({'C01': 4, 'C02': 4, 'C03': 4})


## 7 Select incomplete overdue referrals

Use `2026-10-26` as the assignment snapshot date. A record belongs in `overdue` when:

- `completion_date` is `None`, **and**
- the parsed `due_date` is earlier than the snapshot date.

Parse `due_date` before comparing it with `SNAPSHOT`. Do not compare a date string directly with a Python `date` object.


In [10]:
SNAPSHOT = datetime.strptime("2026-10-26", "%Y-%m-%d").date()

# TODO: build a list named overdue using the two rules above.
overdue = []
for record in cleaned_records:
    if record["completion_date"] is None:
        parsed_due_date = datetime.strptime(record["due_date"], "%Y-%m-%d").date()
        if parsed_due_date < SNAPSHOT:
            overdue.append(record)
print("Overdue incomplete referrals:", len(overdue))
for record in overdue:
    print(record)

assert len(overdue) == 1, "One supplied referral should be incomplete and overdue at the snapshot."


Overdue incomplete referrals: 1
{'referral_id': 'R007', 'patient_id': 'P007', 'clinic_code': 'C01', 'service_type': 'Nutrition', 'priority': 'Routine', 'referral_date': '2026-08-15', 'due_date': '2026-08-29', 'completion_date': None}


## 8 AI-assisted task and verification

Use an approved AI tool for one specified task, such as explaining an error or drafting one standardization function. Record the prompt and relevant response. Then identify the exact test that you used to determine whether the response was correct. If a test failed, explain what you changed.


**AI-use record:**

- Tool: Google Gemini
- Task and prompt: Write a Python function to standardize date strings that might be in either YYYY-MM-DD or MM/DD/YYYY formats using datetime.strptime
- Relevant response: Generated a standardize_date function utilizing a try/except ValueError block to test for both formats and output .isoformat()
- Test and expected result: Executed assert standardize_date("08/03/2026") == "2026-08-03" expecting the function to pass without throwing an AssertionError.
- Actual result: it passed
- Revision and judgment: The AI code was accurate, correctly stripping whitespaces and handling the dual formats as requested.


## 9 Export the cleaned records

Export the same `cleaned_records` list to both formats:

- `Week10_Cleaned_Referrals.csv`
- `Week10_Cleaned_Referrals.json`

For CSV, use `csv.DictWriter`, write the header, and write all records. For JSON, use `json.dump` with `indent=2`. Use UTF-8 encoding for both files.


In [11]:
CSV_OUTPUT = Path("Week10_Cleaned_Referrals.csv")
JSON_OUTPUT = Path("Week10_Cleaned_Referrals.json")

# TODO: write cleaned_records to CSV_OUTPUT.
# Use list(cleaned_records[0].keys()) as the DictWriter fieldnames.
with open(CSV_OUTPUT, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=list(cleaned_records[0].keys()))
    writer.writeheader()
    writer.writerows(cleaned_records)
# TODO: write cleaned_records to JSON_OUTPUT with indent=2.
with open(JSON_OUTPUT, mode="w", encoding="utf-8") as file:
    json.dump(cleaned_records, file, indent=2)

print("CSV created:", CSV_OUTPUT.exists(), CSV_OUTPUT)
print("JSON created:", JSON_OUTPUT.exists(), JSON_OUTPUT)


CSV created: True Week10_Cleaned_Referrals.csv
JSON created: True Week10_Cleaned_Referrals.json
